# Bundle Step 1: Inventory Generation

## Overview

Discover all dashboards and generate comprehensive inventory CSV with metadata.

**What this notebook does:**
1. Loads configuration from databricks.yml parameters
2. Discovers dashboards using optimized system table queries
3. Fetches dashboard names via SDK API
4. Enriches with lineage and audit metadata (17+ fields total)
5. Exports inventory to CSV in Unity Catalog Volume

**Output:** `inventory.csv` in configured volume path

**Next step:** Review inventory in Bundle_02, then run Bundle_03_Export_and_Transform

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U pyyaml pandas databricks-sdk --quiet
dbutils.library.restartPython()

## Cell 1: Load Configuration

In [ ]:
import sys
import os
from pathlib import Path

# Dynamically locate helpers directory
print("=== HELPERS PATH RESOLUTION DEBUG ===")

try:
    # In Databricks workspace/job context
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    print(f"Notebook path (from dbutils): {notebook_path}")
    
    # Add /Workspace prefix if not present
    if not notebook_path.startswith('/Workspace'):
        notebook_path = '/Workspace' + notebook_path
        print(f"Added /Workspace prefix: {notebook_path}")
    
    # Get parent directory of Bundle folder
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    print(f"Notebook dir: {notebook_dir}")
    print(f"Bundle parent: {bundle_parent}")
    
    # Verify helpers exists in bundle_parent
    helpers_path = os.path.join(bundle_parent, 'helpers')
    print(f"\nChecking helpers path: {helpers_path}")
    
    if os.path.exists(helpers_path):
        print(f"  Helpers directory EXISTS")
        init_file = os.path.join(helpers_path, '__init__.py')
        if os.path.exists(init_file):
            print(f"  __init__.py FOUND")
            print(f"  Contents: {os.listdir(helpers_path)[:10]}")
        else:
            print(f"  WARNING: No __init__.py found")
    else:
        print(f"  WARNING: Helpers directory does not exist at {helpers_path}")
    
    # Add BUNDLE PARENT to sys.path (not the helpers dir itself)
    sys_path_entry = bundle_parent
    
except Exception as e:
    print(f"Error in workspace context: {e}")
    import traceback
    traceback.print_exc()
    # Fallback for local execution
    sys_path_entry = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f"Using local fallback: {sys_path_entry}")

sys_path_entry = os.path.normpath(sys_path_entry)

# Add bundle parent to sys.path (NOT the helpers directory)
print(f"\nAdding to sys.path: {sys_path_entry}")
if sys_path_entry not in sys.path:
    sys.path.insert(0, sys_path_entry)
    print(f"  Added to sys.path")
else:
    print(f"  Already in sys.path")

print(f"\nsys.path first 3 entries: {sys.path[:3]}")
print("=== END DEBUG ===\n")

from helpers import (
    set_config,
    get_path,
    write_volume_file,
    ensure_directory_exists,
    create_workspace_client
)





## Cell 0.5: Interactive Testing Setup (Optional)

**ONLY run this cell for interactive/local testing!**

When running as a job, parameters are passed automatically from `databricks.yml`.
When testing interactively, run this cell first to set up test widgets.

In [ ]:
# =============================================================================
# INTERACTIVE TESTING ONLY - Skip this cell when running as a job
# =============================================================================
# Uncomment and customize these values for your environment:

# dbutils.widgets.text("catalog", "your_catalog_name")
# dbutils.widgets.text("volume_base", "/Volumes/your_catalog/your_schema/your_volume")
# dbutils.widgets.text("source_workspace_url", "https://your-workspace.cloud.databricks.com")
# dbutils.widgets.text("inventory_path", "dashboard_inventory")
# dbutils.widgets.text("audit_lookback_days", "90")

print("⚠️  This cell is for interactive testing only!")
print("📝 Uncomment and customize the widget values above for your environment")
print("💡 When running as a job, parameters come from databricks.yml automatically")

In [ ]:
# ============================================================================
# CONFIGURATION FROM PARAMETERS (databricks.yml)
# ============================================================================
# This notebook reads configuration ONLY from job parameters (widgets)
# All configuration is in databricks.yml
#
# For interactive testing, run Cell 0.5 first to set up test widgets

print("📋 Loading configuration from parameters...\n")

# Read all parameters from widgets
catalog = dbutils.widgets.get('catalog')
volume_base = dbutils.widgets.get('volume_base')
source_workspace_url = dbutils.widgets.get('source_workspace_url')
inventory_path_rel = dbutils.widgets.get('inventory_path')
audit_lookback_days = int(dbutils.widgets.get('audit_lookback_days'))

# Build full inventory path
inventory_path = f"{volume_base}/{inventory_path_rel}"

# Build config dict for helper functions
config = {
    'dashboard_selection': {
        'catalog_filter': {
            'catalog': catalog
        }
    },
    'paths': {
        'volume_base': volume_base,
        'inventory': inventory_path_rel
    },
    'source': {
        'workspace_url': source_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'inventory': {
        'audit_lookback_days': audit_lookback_days
    }
}

# Cache config for helper functions
set_config(config)

print("✅ Configuration loaded from parameters")
print(f"   Catalog: {catalog}")
print(f"   Volume: {volume_base}")
print(f"   Inventory path: {inventory_path}")
print(f"   Audit lookback: {audit_lookback_days} days\n")

## Cell 2: Get Workspace ID

In [ ]:
# Get workspace ID - try multiple methods for serverless compatibility
workspace_id = None

# Method 1: Try dbutils (works in most contexts including serverless)
try:
    workspace_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get('orgId').get()
    print(f"✅ Workspace ID (from dbutils): {workspace_id}")
except:
    pass

# Method 2: Try spark.conf (works in standard clusters)
if not workspace_id:
    try:
        workspace_id = spark.conf.get('spark.databricks.clusterUsageTags.orgId')
        print(f"✅ Workspace ID (from spark.conf): {workspace_id}")
    except:
        pass

# Method 3: Query from system table as fallback
if not workspace_id:
    try:
        result = spark.sql("SELECT DISTINCT workspace_id FROM system.access.table_lineage LIMIT 1").collect()
        if result:
            workspace_id = result[0].workspace_id
            print(f"✅ Workspace ID (from system table): {workspace_id}")
    except:
        pass

if not workspace_id:
    raise ValueError("Could not determine workspace ID. Please check your Databricks configuration.")

print(f"\n✅ Using Workspace ID: {workspace_id}")

## Cell 3: Query 1 - Discover Dashboards

Find all dashboards using system.access.table_lineage

In [ ]:
# Main discovery query
print(f"🔍 Discovering dashboards for catalog: {catalog}\n")

query = f"""
SELECT DISTINCT 
    entity_id as dashboard_id,
    COUNT(*) as reference_count
FROM system.access.table_lineage
WHERE workspace_id = {workspace_id}
  AND source_table_catalog = '{catalog}'
  AND entity_type = 'DASHBOARD_V3'
  AND entity_id IS NOT NULL
GROUP BY entity_id
ORDER BY reference_count DESC
"""

result = spark.sql(query)
dashboard_count = result.count()

print(f"✅ Found {dashboard_count} dashboards\n")
display(result)

# Save to temporary view for reuse
result.createOrReplaceTempView("discovered_dashboards")
print(f"\n✅ Saved to temp view: discovered_dashboards")

## Cell 4: Fetch Dashboard Names (SDK API)

System tables don't contain dashboard names - we need to fetch them via SDK API

In [ ]:
# Connect to source workspace
print("🔗 Connecting to workspace to fetch dashboard names...\n")
client = create_workspace_client('source')


# Get dashboard IDs from temp view
dashboard_ids = [row.dashboard_id for row in spark.sql("SELECT dashboard_id FROM discovered_dashboards").collect()]


print(f"📋 Fetching names for {len(dashboard_ids)} dashboards...\n")

# Fetch names via API
dashboard_names = []
failed_count = 0

for i, dash_id in enumerate(dashboard_ids, 1):
    try:
        
        dash = client.lakeview.get(dash_id)
        name = dash.display_name or f"Dashboard_{dash_id}"
        
        
        dashboard_names.append({
            'dashboard_id': dash_id,
            'dashboard_name': name
        })
        if i % 10 == 0:
            print(f"  Progress: {i}/{len(dashboard_ids)}")
    except Exception as e:
        error_str = str(e)
        error_type = type(e).__name__
        
        
        print(f"  ⚠️  Failed to get name for {dash_id}: {e}")
        dashboard_names.append({
            'dashboard_id': dash_id,
            'dashboard_name': f"Dashboard_{dash_id}"
        })
        failed_count += 1


print(f"\n✅ Fetched {len(dashboard_names) - failed_count}/{len(dashboard_ids)} dashboard names")
if failed_count > 0:
    print(f"⚠️  {failed_count} dashboards used fallback names")

# Create temp view with names
import pandas as pd
names_df = spark.createDataFrame(pd.DataFrame(dashboard_names))
names_df.createOrReplaceTempView("dashboard_names")

print(f"\n✅ Created temp view: dashboard_names")
display(names_df)

## Cell 5: Aggregate Lineage Overview

In [ ]:
# Get comprehensive lineage for all discovered dashboards
print("📊 Aggregating lineage metadata...\n")

lineage_query = f"""
SELECT 
    entity_id as dashboard_id,
    COUNT(DISTINCT source_table_catalog) as catalog_count,
    COUNT(DISTINCT source_table_name) as table_count,
    COUNT(DISTINCT source_table_full_name) as unique_tables,
    COUNT(*) as total_lineage_entries
FROM system.access.table_lineage
WHERE workspace_id = {workspace_id}
    AND source_table_catalog = '{catalog}'
    AND entity_type = 'DASHBOARD_V3'
    AND entity_id IS NOT NULL
GROUP BY entity_id
ORDER BY table_count DESC
"""

lineage_result = spark.sql(lineage_query)
print(f"✅ Lineage metadata for {lineage_result.count()} dashboards\n")
display(lineage_result.limit(10))

## Cell 6: Build Complete Inventory

Build comprehensive inventory with optimized SQL (3-step process):
- Step 1: Join discovered dashboards with names and lineage data
- Step 2: Aggregate audit/access data
- Step 3: Join lineage + audit to create complete inventory

In [ ]:
# Build complete inventory in 3 steps (optimized for performance)
print("🔨 Building complete dashboard inventory...\n")

# Get audit lookback days from config (default 90)
audit_days = config.get('inventory', {}).get('audit_lookback_days', 90)

# ============================================================================
# STEP 1/3: Build lineage_data temp view (with names)
# ============================================================================
print("Step 1/3: Building lineage data with names...")
import time
start_time = time.time()

lineage_query = f"""
SELECT 
    dd.dashboard_id,
    dn.dashboard_name,
    dd.reference_count,
    tl.entity_type,
    COUNT(DISTINCT tl.source_table_catalog) as catalog_count,
    COUNT(DISTINCT tl.source_table_name) as table_count,
    COUNT(DISTINCT tl.source_table_full_name) as unique_tables,
    COUNT(*) as lineage_entries,
    COLLECT_SET(tl.source_table_catalog) as catalogs_used,
    COLLECT_SET(tl.source_table_full_name) as tables_used
FROM discovered_dashboards dd
INNER JOIN dashboard_names dn ON dd.dashboard_id = dn.dashboard_id
LEFT JOIN system.access.table_lineage tl
    ON tl.entity_id = dd.dashboard_id
    AND tl.workspace_id = {workspace_id}
    AND tl.entity_type = 'DASHBOARD_V3'
GROUP BY dd.dashboard_id, dn.dashboard_name, dd.reference_count, tl.entity_type
"""

lineage_df = spark.sql(lineage_query)
lineage_df.createOrReplaceTempView("lineage_data")
lineage_count = lineage_df.count()
step1_time = time.time() - start_time
print(f"  ✓ Created lineage_data ({lineage_count} rows) in {step1_time:.1f}s\n")

# ============================================================================
# STEP 2/3: Build audit_data temp view
# ============================================================================
print("Step 2/3: Building audit data...")
start_time = time.time()

audit_query = f"""
SELECT 
    request_params.dashboard_id as dashboard_id,
    MAX(event_time) as last_accessed,
    MIN(event_time) as first_accessed,
    COUNT(*) as total_access_count,
    COUNT(DISTINCT user_identity.email) as unique_users,
    COUNT(DISTINCT DATE(event_time)) as unique_access_days,
    DATEDIFF(CURRENT_DATE(), MAX(DATE(event_time))) as days_since_last_access
FROM system.access.audit
WHERE action_name IN ('getDashboard', 'getPublishedDashboard', 'getDashboardSubscription')
  AND event_date >= CURRENT_DATE() - INTERVAL {audit_days} DAYS
  AND request_params.dashboard_id IN (SELECT dashboard_id FROM discovered_dashboards)
GROUP BY request_params.dashboard_id
"""

audit_df = spark.sql(audit_query)
audit_df.createOrReplaceTempView("audit_data")
audit_count = audit_df.count()
step2_time = time.time() - start_time
print(f"  ✓ Created audit_data ({audit_count} rows) in {step2_time:.1f}s\n")

# ============================================================================
# STEP 3/3: Join temp views to create complete_inventory
# ============================================================================
print("Step 3/3: Joining lineage + audit data...")
start_time = time.time()

# Check if we should include large fields
include_catalogs = config.get('inventory', {}).get('export', {}).get('include_catalogs_used', True)
include_tables = config.get('inventory', {}).get('export', {}).get('include_tables_used', False)

catalogs_field = "ld.catalogs_used," if include_catalogs else ""
tables_field = "ld.tables_used," if include_tables else ""

final_query = f"""
SELECT 
    ld.dashboard_id,
    ld.dashboard_name,
    ld.reference_count,
    ld.entity_type,
    COALESCE(ld.catalog_count, 0) as catalog_count,
    COALESCE(ld.table_count, 0) as table_count,
    COALESCE(ld.unique_tables, 0) as unique_tables,
    COALESCE(ld.lineage_entries, 0) as lineage_entries,
    {catalogs_field}
    {tables_field}
    ad.last_accessed,
    ad.first_accessed,
    COALESCE(ad.total_access_count, 0) as total_access_count,
    COALESCE(ad.unique_users, 0) as unique_users,
    COALESCE(ad.unique_access_days, 0) as unique_access_days,
    COALESCE(ad.days_since_last_access, 9999) as days_since_last_access,
    CASE 
        WHEN ld.table_count > 20 THEN 'High'
        WHEN ld.table_count > 10 THEN 'Medium'
        ELSE 'Low'
    END as complexity,
    CASE 
        WHEN ad.last_accessed IS NULL THEN 'Inactive'
        WHEN ad.days_since_last_access <= 7 THEN 'Very Active'
        WHEN ad.days_since_last_access <= 30 THEN 'Active'
        WHEN ad.days_since_last_access <= 60 THEN 'Moderate'
        ELSE 'Low Activity'
    END as activity_level
FROM lineage_data ld
LEFT JOIN audit_data ad
    ON ld.dashboard_id = ad.dashboard_id
ORDER BY ld.table_count DESC, ad.total_access_count DESC
"""

complete_inventory_df = spark.sql(final_query)
complete_inventory_df.createOrReplaceTempView("complete_inventory")

inventory_count = complete_inventory_df.count()
step3_time = time.time() - start_time
total_time = step1_time + step2_time + step3_time

print(f"  ✓ Created complete_inventory ({inventory_count} rows) in {step3_time:.1f}s")
print(f"\n{'='*70}")
print(f"✅ Built complete inventory with {inventory_count} dashboards")
print(f"📊 Total fields: {len(complete_inventory_df.columns)}")
print(f"⏱️  Total time: {total_time:.1f}s")
print(f"{'='*70}\n")

# Display sample
display(complete_inventory_df.limit(20))

## Cell 7: Export Inventory to CSV in Volume

Save complete inventory to Unity Catalog Volume for use by Bundle_01

In [ ]:
# ============================================================================
# STEP 1: Ensure Volume Exists
# ============================================================================
print("🔍 Checking volume structure...\n")

# Parse volume components from volume_base
# Example: /Volumes/archana_krish_fe_dsa/vizient_deep_dive/dashboard_migration
volume_parts = volume_base.strip('/').split('/')
if len(volume_parts) >= 4 and volume_parts[0] == 'Volumes':
    catalog = volume_parts[1]
    schema = volume_parts[2]
    volume_name = volume_parts[3]
    
    print(f"📦 Volume Configuration:")
    print(f"   Catalog: {catalog}")
    print(f"   Schema: {schema}")
    print(f"   Volume: {volume_name}")
    print(f"   Base Path: {volume_base}\n")
    
    # Check if volume exists
    try:
        existing_volumes = spark.sql(f"""
            SHOW VOLUMES IN {catalog}.{schema}
        """).collect()
        
        volume_exists = any(v.volume_name == volume_name for v in existing_volumes)
        
        if volume_exists:
            print(f"✅ Volume '{volume_name}' already exists\n")
        else:
            print(f"⚠️  Volume '{volume_name}' does NOT exist")
            print(f"📦 Creating volume: {catalog}.{schema}.{volume_name}...\n")
            
            spark.sql(f"""
                CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_name}
                COMMENT 'Dashboard migration artifacts - inventory, exports, transforms, bundles'
            """)
            
            print(f"✅ Created volume: {catalog}.{schema}.{volume_name}\n")
    
    except Exception as e:
        print(f"⚠️  Could not verify/create volume: {e}")
        print(f"   Assuming volume exists and continuing...\n")
else:
    print(f"⚠️  Could not parse volume path: {volume_base}")
    print(f"   Assuming volume exists and continuing...\n")

# ============================================================================
# STEP 2: Create ALL Required Directories
# ============================================================================
print("📁 Creating migration directories...\n")

# Create raw inventory directory
ensure_directory_exists(inventory_path)
print(f"✅ Raw inventory directory: {inventory_path}")

# Create approved inventory directory (empty, ready for manual upload)
approved_inventory_path = f"{volume_base}/dashboard_inventory_approved"
ensure_directory_exists(approved_inventory_path)
print(f"✅ Approved inventory directory: {approved_inventory_path}")
print(f"   (Empty - ready for your approved CSV upload after review)\n")

# ============================================================================
# STEP 3: Export Inventory to CSV
# ============================================================================
# Convert to pandas and create CSV
print("📥 Converting to CSV...")
inventory_pandas = complete_inventory_df.toPandas()
csv_content = inventory_pandas.to_csv(index=False)

# Write to volume
csv_file_path = f"{inventory_path}/inventory.csv"
write_volume_file(csv_file_path, csv_content, overwrite=True)

print(f"\n{'='*70}")
print(f"✅ INVENTORY EXPORTED SUCCESSFULLY")
print(f"{'='*70}")
print(f"📁 Location: {csv_file_path}")
print(f"📊 Total dashboards: {len(inventory_pandas)}")
print(f"📋 Fields: {len(inventory_pandas.columns)}")
print(f"💾 Size: {len(csv_content) / 1024:.1f} KB")
print(f"{'='*70}\n")

# Show summary statistics
print("📊 Inventory Summary:")
print(f"  Entity Types: {inventory_pandas['entity_type'].nunique()}")
print(f"  Avg Tables/Dashboard: {inventory_pandas['table_count'].mean():.1f}")
print(f"  Max Tables: {inventory_pandas['table_count'].max()}")
print(f"\n  Complexity:")
print(inventory_pandas['complexity'].value_counts().to_string())
print(f"\n  Activity:")
print(inventory_pandas['activity_level'].value_counts().to_string())
print(f"\n{'='*70}")
print(f"🎯 NEXT STEP: REVIEW & APPROVE (REQUIRED)")
print(f"{'='*70}")
print(f"\nStep 2: Run Bundle/Bundle_02_Review_and_Approve_Inventory.ipynb")
print(f"\nInstructions:")
print(f"  1. Open the notebook in Databricks UI")
print(f"  2. Review inventory statistics (Cells 2-4)")
print(f"  3. Customize filters if needed (Cell 5)")
print(f"  4. Review approved list (Cell 6)")
print(f"  5. Type CONFIRM to upload (Cell 7)")
print(f"  6. Verify upload (Cell 8)")
print(f"\nAfter approval, run Step 3:")
print(f"  databricks bundle run export_transform -t dev")
print(f"{'='*70}")